**Eurovision Voting Blocs Across Eras (1975–2024)**

# 02 - Network Construction

**Objectives**
- Merge the raw vote tables into one clean table of Grand Final votes (1975–2024)
- Turn the votes of each period into a directed, weighted network of "who favours whom"
- Build binary networks (an edge either exists or not) that keep only *stable* ties, for the measures that ignore weights
- Rebuild the networks with alternative settings (other values of k, rank-based weights, P1 (1975–1996) divided at 1993), so that notebook 03 can check that the conclusions do not depend on one setting
- Test how the method's constants (k, τ, minimum years) change the results and explain the values chosen
- Build separate jury and televote networks for P4
- Build the country-pair attributes (shared border, shared language) from CEPII
- Export everything to `data/processed/` and the dataset summary table to `paper/`

**Input:** files in `data/raw/` (notebook 01)

> Run this notebook from the `src/` folder. All later notebooks read only its outputs.

In [2]:
import os

import numpy as np
import pandas as pd
import pyreadstat
import networkx as nx

from helper_functions import compute_period_network, compute_period_rank_network, binarize_network

## 1. Setup

### 1.1. Paths

In [3]:
BASE_DIR = os.path.abspath("..")
DATA_RAW = os.path.join(BASE_DIR, "data", "raw")
DATA_PROC = os.path.join(BASE_DIR, "data", "processed")
DATA_ROBUST = os.path.join(DATA_PROC, "robustness")
DATA_CHANNELS = os.path.join(DATA_PROC, "p4_channels")
PAPER_DIR = os.path.join(BASE_DIR, "paper")

for d in (DATA_PROC, DATA_ROBUST, DATA_CHANNELS, PAPER_DIR):
    os.makedirs(d, exist_ok=True)

### 1.2. Periods

The 50 years of the contest are split at the main changes in who decides each country's points (national juries, televote, or both). The project compares voting blocs across these eras, so each period gets its own network.

| Period | Years | Finals | Who decides each country's points |
|---|---|---|---|
| **P1** | 1975–1996 | 22 | A national jury (a panel of people in each country) |
| **P2** | 1997–2008 | 12 | The public, by phone (televote): introduced in 5 countries in 1997 and used by most countries from 1998 |
| **P3** | 2009–2015 | 7 | A national jury and the televote together: each ranks the songs and the two rankings are merged 50/50 into one set of points |
| **P4** | 2016–2024 | 8 | A national jury and the televote separately: each gives its own full set of 12-10-8-…-1 points, so a country can receive up to 24 points from another |

The points themselves also change between periods (since 2016 a country can give up to 24 points to another, before only 12), so raw points cannot be compared. Section 3 explains how this bias is avoided.

In [4]:
ERAS = {
    "P1": (1975, 1996),
    "P2": (1997, 2008),
    "P3": (2009, 2015),
    "P4": (2016, 2024),
}

### 1.3. Country Codes

The vote tables use two-letter codes (`se`, `gr`, …) and CEPII uses three-letter ISO codes (`SWE`, `GRC`, …). Every country is converted to the three-letter code. Two dissolved states keep their own code instead of being merged into a successor state: Yugoslavia (`YUG`, until 1992) and Serbia and Montenegro (`SCG`, 2004–2006).

In [5]:
ISO2_TO_ISO3 = {
    "AL": "ALB", "AD": "AND", "AM": "ARM", "AU": "AUS", "AT": "AUT", "AZ": "AZE",
    "BY": "BLR", "BE": "BEL", "BA": "BIH", "BG": "BGR", "HR": "HRV", "CY": "CYP",
    "CZ": "CZE", "DK": "DNK", "EE": "EST", "FI": "FIN", "FR": "FRA", "GE": "GEO",
    "DE": "DEU", "GR": "GRC", "HU": "HUN", "IS": "ISL", "IE": "IRL", "IL": "ISR",
    "IT": "ITA", "LV": "LVA", "LT": "LTU", "LU": "LUX", "MT": "MLT", "MD": "MDA",
    "MC": "MCO", "ME": "MNE", "MA": "MAR", "NL": "NLD", "MK": "MKD", "NO": "NOR",
    "PL": "POL", "PT": "PRT", "RO": "ROU", "RU": "RUS", "SM": "SMR", "RS": "SRB",
    "CS": "SCG", "SK": "SVK", "SI": "SVN", "ES": "ESP", "SE": "SWE", "CH": "CHE",
    "TR": "TUR", "UA": "UKR", "GB": "GBR", "UK": "GBR", "YU": "YUG"
}

## 2. Load and Merge the Grand Final Votes

Only Grand Final votes are used. Semi-finals are left out because each country only votes in its own semi-final, so many pairs of countries never have the chance to vote for each other there. In the final, every participant (including countries eliminated in the semi-finals) votes for every finalist.

- 1975–2023 come from the Spijkervet dataset and 2024 from EurovisionDataset.
- The "Rest of the World" online vote (`WLD`, since 2023) is removed.

In [6]:
df_spijk = pd.read_csv(os.path.join(DATA_RAW, "votes_spijkervet.csv"))
df_josago = pd.read_csv(os.path.join(DATA_RAW, "votes_josago.csv"))

In [7]:
# keep grand final votes only
df_spijk_final = df_spijk[(df_spijk["year"] >= 1975) & (df_spijk["year"] <= 2023) & (df_spijk["round"] == "final")].copy()
df_josago_final = df_josago[(df_josago["year"] >= 2024) & (df_josago["round"] == "final")].copy()

cols = ["year", "from_country_id", "to_country_id", "total_points"]
df_all = pd.concat([df_spijk_final[cols], df_josago_final[cols]], ignore_index=True)

In [8]:
# clean country codes and remove non-countries
df_all["from_country_id"] = df_all["from_country_id"].astype(str).str.strip().str.upper()
df_all["to_country_id"] = df_all["to_country_id"].astype(str).str.strip().str.upper()

df_all = df_all[~df_all["from_country_id"].isin(["WLD", "ROW"])]
df_all = df_all[~df_all["to_country_id"].isin(["WLD", "ROW"])]

In [9]:
# convert to three-letter codes
df_all["from_iso3"] = df_all["from_country_id"].map(ISO2_TO_ISO3)
df_all["to_iso3"] = df_all["to_country_id"].map(ISO2_TO_ISO3)

unmapped_from = df_all[df_all["from_iso3"].isna()]["from_country_id"].unique()
unmapped_to = df_all[df_all["to_iso3"].isna()]["to_country_id"].unique()
if len(unmapped_from) > 0 or len(unmapped_to) > 0:
    raise ValueError(f"Unmapped country codes: from={unmapped_from}, to={unmapped_to}")

### 2.1. Consistency Check Between the Two Sources

Both sources cover 1975–2023, but we only take 2024 from EurovisionDataset. Comparing them on the years they share is a direct check that the 2024 records are compatible with the rest of the data.

In [10]:
def final_rows(df):
    out = df[(df["round"] == "final") & df["year"].between(1975, 2023)].copy()
    out["f"], out["t"] = out["from_country_id"].str.upper(), out["to_country_id"].str.upper()
    return out[["year", "f", "t", "total_points"]]

overlap = final_rows(df_spijk).merge(final_rows(df_josago), on=["year", "f", "t"], suffixes=("_spijkervet", "_josago"))
mismatches = (overlap["total_points_spijkervet"] != overlap["total_points_josago"]).sum()
print(f"Shared vote records 1975–2023: {len(overlap)} | records with different points: {mismatches}")

Shared vote records 1975–2023: 32031 | records with different points: 0


### 2.2. Final Vote Table

In [11]:
df_all["total_points"] = pd.to_numeric(df_all["total_points"], errors="coerce").fillna(0)
df_votes = df_all

print(f"Loaded {len(df_votes)} Grand Final vote rows across years {df_votes['year'].min()} to {df_votes['year'].max()}.")
all_nodes = sorted(set(df_votes["from_iso3"].unique()) | set(df_votes["to_iso3"].unique()))
print(f"Total unique countries across all eras: {len(all_nodes)}")
df_votes.head()

Loaded 34014 Grand Final vote rows across years 1975 to 2024.
Total unique countries across all eras: 52


,year,from_country_id,to_country_id,total_points,from_iso3,to_iso3
0,1975,BE,NL,3,BEL,NLD
1,1975,BE,GB,10,BEL,GBR
2,1975,BE,IT,6,BEL,ITA
3,1975,BE,FR,2,BEL,FRA
4,1975,BE,LU,0,BEL,LUX


## 3. From Votes to Networks

### 3.1. The Idea

A network is a set of nodes connected by edges. Here:

- each node is a country;
- each edge A → B is directed (A voting for B is not the same as B voting for A);
- each edge has a weight that says how strong the relation is.

As mentioned earlier, raw points cannot be compared across periods, so we only use the ranking inside each vote:

1. **Within each final (year *t*)**, draw an edge A → B if B is among the *k* countries that received the most points from A. With *k* = 3, these are normally the countries that got 12, 10 and 8 points.
2. **Over the whole period *p***, the weight of A → B is the share of years in which A put B in its top *k*, counting only the years in which A voted and B competed in the final:

$$w_{AB}^{(p)} = \frac{\text{years in which B was in A's top } k}{\text{years in which A voted and B was a finalist}} \in [0, 1]$$

Example (P2, 12 finals): Cyprus could vote for Greece in 9 finals and put it in its top 3 in all 9, so $w = 9/9 = 1.0$. Malta could vote for the United Kingdom in 12 finals and did so in 4, so $w = 4/12 = 0.33$.

Years in which one of the two countries was absent do not count against the pair: they are simply left out of the denominator.

**Ties.** In P1–P3 a voter never gives the same points to two countries, so the top 3 is always exactly three countries. In P4 we sum jury and televote points, and two countries can end with the same total. The function then keeps every country tied with the third place (see 3.4).

### 3.2. The Network-Building Function

`compute_period_network` (in `helper_functions.py`) builds the network of one period as follow:

**Inputs**

| Argument | Meaning |
|---|---|
| `df_votes` | Grand Final vote table, with columns `year`, `from_iso3`, `to_iso3`, `total_points` |
| `period_name` | Name of the period (e.g. `"P1"`); not used in the computation |
| `year_min`, `year_max` | First and last year of the period (both included) |
| `k` | How many top-ranked countries of each vote count as "favoured" (default is 3) |

**Outputs** (in this order)

| Output | Meaning |
|---|---|
| `adj_matrix` | Weighted adjacency matrix: one row and one column per country; cell *(A, B)* holds $w_{AB}$ |
| `edgelist_df` | One row per edge with $w_{AB} > 0$: `from`, `to`, `weight`, `topk_years`, `co_participation_years`, sorted by weight |
| `nodes` | Sorted list of the countries that voted or competed in the period |
| `topk_years` matrix | Cell *(A, B)*: number of years in which A put B in its top *k* (numerator of $w_{AB}$) |
| `co_years` matrix | Cell *(A, B)*: number of years in which A voted and B was a finalist (denominator of $w_{AB}$) |

### 3.3. Build the Main Networks (k = 3)

In [12]:
main_results = {}
period_nodes = {}
for period, (y0, y1) in ERAS.items():
    adj_topk3, edgelist_topk3, nodes, mat_topk_years, mat_co_years = compute_period_network(df_votes, period, y0, y1, k=3)
    main_results[period] = (adj_topk3, edgelist_topk3, nodes, mat_topk_years, mat_co_years)
    period_nodes[period] = nodes
    print(f"{period} ({y0}–{y1}): {len(nodes)} nodes, {len(edgelist_topk3)} non-zero directed edges.")

P1 (1975–1996): 35 nodes, 518 non-zero directed edges.
P2 (1997–2008): 48 nodes, 615 non-zero directed edges.
P3 (2009–2015): 47 nodes, 536 non-zero directed edges.
P4 (2016–2024): 45 nodes, 686 non-zero directed edges.


In [13]:
for period, (adj_topk3, edgelist_topk3, *_) in main_results.items():
    adj_topk3.to_csv(os.path.join(DATA_PROC, f"{period}_adj_topk3.csv"))
    edgelist_topk3.to_csv(os.path.join(DATA_PROC, f"{period}_edgelist_topk3.csv"), index=False)

### 3.4. A Worked Example

A subset of the P2 edge list:

In [14]:
e = main_results["P2"][1]
e[((e["from"] == "CYP") & (e["to"] == "GRC")) | ((e["from"] == "GRC") & (e["to"] == "CYP")) | ((e["from"] == "MLT") & (e["to"] == "GBR"))]

,from,to,weight,topk_years,co_participation_years
33,GRC,CYP,1.00000,6,6
57,CYP,GRC,1.00000,9,9
229,MLT,GBR,0.33333,4,12


### 3.5. How Often Is the Top 3 Larger Than Three?

Share of votes (one voter in one year) whose top 3 contains more than three countries because of ties. This only happens in P4, where jury and televote points are summed.

In [15]:
for period, (y0, y1) in ERAS.items():
    df_p = df_votes[df_votes["year"].between(y0, y1)]
    sizes = []
    for _, vg in df_p.groupby(["year", "from_iso3"]):
        pts = vg["total_points"].sort_values(ascending=False).values
        cutoff = pts[2]
        sizes.append((vg["total_points"] >= cutoff).sum() if cutoff > 0 else (vg["total_points"] > 0).sum())
    sizes = np.array(sizes)
    print(f"{period}: {np.mean(sizes > 3):.1%} of votes have a top 3 with more than three countries")

P1: 0.0% of votes have a top 3 with more than three countries
P2: 0.0% of votes have a top 3 with more than three countries
P3: 0.0% of votes have a top 3 with more than three countries
P4: 20.6% of votes have a top 3 with more than three countries


## 4. Binary Networks of Stable Ties

Some measures (density, reciprocity, triad census) only ask whether an edge exists, not how strong it is. We therefore keep only "stable ties". 

A → B is kept when both conditions hold:

1. **Share of years (τ):** B was in A's top 3 in at least τ of the possible years;
2. **Repeated support:** this happened in at least **2** different years.

The second condition removes a misleading case: a country that took part only once gets $w = 1/1 = 1.0$ from a single vote.

| Version | τ | Min. years | File | Used for |
|---|---|---|---|---|
| **Main** | 0.5 | 2 | `{period}_adj_binary_t05.csv` | density, reciprocity, triads (notebook 03) |
| **Robustness** | 0.33 | 2 | `robustness/{period}_adj_binary_t033.csv` | the same measures, as a check |
| **Reference** | 0.5 | — | `robustness/{period}_adj_binary_t05_unconstrained.csv` | not used in analyses; shows the effect of the 2-year rule |

Examples (P2): Cyprus → Greece (9/9) is a stable tie in both versions; Malta → United Kingdom (4/12 = 0.33) only in the robustness version.

### 4.1. Main Binary Networks (τ = 0.5, at least 2 years)

In [40]:
for period, (adj_topk3, _, _, mat_topk_years, _) in main_results.items():
    adj_bin_t05 = binarize_network(adj_topk3, mat_topk_years, tau=0.5, min_years=2)
    adj_bin_t05.to_csv(os.path.join(DATA_PROC, f"{period}_adj_binary_t05.csv"))
    print(f"{period}: {adj_bin_t05.values.sum()} stable ties out of {(adj_topk3 > 0).values.sum()} edges")

P1: 23 stable ties out of 518 edges
P2: 112 stable ties out of 615 edges
P3: 99 stable ties out of 536 edges
P4: 101 stable ties out of 686 edges


### 4.2. Robustness Binary Networks (τ = 0.33, at least 2 years)

In [ ]:
for period, (adj_topk3, _, _, mat_topk_years, _) in main_results.items():
    adj_bin_t033 = binarize_network(adj_topk3, mat_topk_years, tau=0.33, min_years=2)
    adj_bin_t033.to_csv(os.path.join(DATA_ROBUST, f"{period}_adj_binary_t033.csv"))
    print(f"{period}: {adj_bin_t033.values.sum()} stable ties out of {(adj_topk3 > 0).values.sum()} edges")

### 4.3. Reference: Without the 2-Year Rule

In [ ]:
for period, (adj_topk3, _, _, mat_topk_years, _) in main_results.items():
    # exported only as a reference: no minimum number of years
    adj_bin_t05_unconstrained = binarize_network(adj_topk3, mat_topk_years, tau=0.5)
    adj_bin_t05_unconstrained.to_csv(os.path.join(DATA_ROBUST, f"{period}_adj_binary_t05_unconstrained.csv"))
    print(f"{period}: {adj_bin_t05_unconstrained.values.sum()} ties out of {(adj_topk3 > 0).values.sum()} edges")

## 5. Alternative Networks: Exploring Sensitivity to Network Definitions

The main networks of Section 3 depend on three choices. This section changes **one choice at a time** and rebuilds the networks, so that notebook 03 can compare them with the main networks. If the voting blocs stay the same, the conclusions do not depend on these choices.

| Choice in Section 3 | Alternative tested here | Section |
|---|---|---|
| A country is "favoured" when it is in the voter's top **k = 3** | k = 2 and k = 4 | 5.1 |
| The weight counts only *whether* B was in A's top 3 | The weight also counts *how high* B was ranked | 5.2 |
| P1 is one period (1975–1996) | P1 divided at 1993 | 5.3 |

### 5.1. Other Values of k (instead of k = 3)

- **k = 2:** only the 12- and 10-point countries count (the strongest preferences);
- **k = 4:** 12, 10, 8 and 7 points.

In [19]:
alt_k_results = {}
for k_val in [2, 4]:
    for period, (y0, y1) in ERAS.items():
        adj_k, edgelist_k, _, topk_years_k, _ = compute_period_network(df_votes, period, y0, y1, k=k_val)
        alt_k_results[(period, k_val)] = (adj_k, topk_years_k)
        adj_k.to_csv(os.path.join(DATA_ROBUST, f"{period}_adj_topk{k_val}.csv"))
        edgelist_k.to_csv(os.path.join(DATA_ROBUST, f"{period}_edgelist_topk{k_val}.csv"), index=False)
        print(f"  Exported robustness/{period}_adj_topk{k_val}.csv & edgelist ({len(edgelist_k)} edges)")

  Exported robustness/P1_adj_topk2.csv & edgelist (424 edges)
  Exported robustness/P2_adj_topk2.csv & edgelist (435 edges)
  Exported robustness/P3_adj_topk2.csv & edgelist (385 edges)
  Exported robustness/P4_adj_topk2.csv & edgelist (502 edges)
  Exported robustness/P1_adj_topk4.csv & edgelist (593 edges)
  Exported robustness/P2_adj_topk4.csv & edgelist (774 edges)
  Exported robustness/P3_adj_topk4.csv & edgelist (674 edges)
  Exported robustness/P4_adj_topk4.csv & edgelist (847 edges)


### 5.2. Rank-Based Weights (instead of top-k share)

Instead of "in the top *k* or not", every position gets a score: 1st place (12 points) → 1.0, 2nd → 0.9, …, 10th (1 point) → 0.1, and 0 otherwise. The period weight is the average score A gave B over the years they could vote. This keeps information about *how high* B was ranked, not only whether it made the top 3. Ties (possible in P4) receive the average score of the positions they share.

In [21]:
for period, (y0, y1) in ERAS.items():
    adj_rank = compute_period_rank_network(df_votes, period, y0, y1)
    adj_rank.to_csv(os.path.join(DATA_ROBUST, f"{period}_adj_rank.csv"))
    nonzero_rank = (adj_rank.values > 0).sum()
    print(f"  Exported robustness/{period}_adj_rank.csv ({nonzero_rank} non-zero entries)")

  Exported robustness/P1_adj_rank.csv (803 non-zero entries)
  Exported robustness/P2_adj_rank.csv (1395 non-zero entries)
  Exported robustness/P3_adj_rank.csv (1323 non-zero entries)
  Exported robustness/P4_adj_rank.csv (1425 non-zero entries)


### 5.3. P1 Divided at 1993 (instead of one period)

P1 is long (22 finals) and the Eastern European countries only enter in 1993–1994. Notebook 03 checks whether the blocs of P1 exist before that entry by analysing 1975–1992 (P1a) and 1993–1996 (P1b) separately.

In [22]:
for name, (y0, y1) in {"P1a": (1975, 1992), "P1b": (1993, 1996)}.items():
    adj_split = compute_period_network(df_votes, name, y0, y1, k=3)[0]
    adj_split.to_csv(os.path.join(DATA_ROBUST, f"{name}_adj_topk3.csv"))
    print(f"  Exported robustness/{name}_adj_topk3.csv ({len(adj_split)} countries)")

  Exported robustness/P1a_adj_topk3.csv (25 countries)
  Exported robustness/P1b_adj_topk3.csv (32 countries)


## 6. Choosing the Constants

The method has three constants. This section shows what happens to the binary networks when each of them changes.

| Constant | Meaning |
|---|---|
| **k** | how many top-ranked countries count as "favoured" in each vote |
| **τ** | minimum share of possible years in which A must have favoured B |
| **min. years** | minimum number of years in which A must have favoured B |

Two measures are compared:

- **Stable ties** and **isolated countries** (countries with no stable tie at all): show how full or empty the network becomes.
- **Reciprocity**: of all stable ties A → B, the share for which B → A also exists. It answers "are alliances mutual?".
- **Reciprocity vs random**: reciprocity divided by the **density** (the share of all possible edges that exist). In a random network with the same density, a tie is returned with probability equal to the density, so a ratio of 5 means mutual ties are five times more frequent than chance.

### 6.1. Sensitivity Grid

In [23]:
all_topk = {(p, 3): (res[0], res[3]) for p, res in main_results.items()}
all_topk.update(alt_k_results)

rows = []
for (period, k_val), (adj, topk_years) in sorted(all_topk.items()):
    for tau in [0.33, 0.5]:
        for min_years in [1, 2, 3, 4]:
            A = ((adj >= tau) & (topk_years >= min_years)).astype(int)
            A = A.where(~np.eye(len(A), dtype=bool), 0)
            G = nx.from_pandas_adjacency(A, create_using=nx.DiGraph)
            dens = nx.density(G)
            rec = nx.reciprocity(G) if G.number_of_edges() else np.nan
            rows.append({"period": period, "k": k_val, "tau": tau, "min_years": min_years,
                         "stable_ties": G.number_of_edges(),
                         "isolated": len(list(nx.isolates(G))),
                         "reciprocity": rec,
                         "reciprocity_vs_random": rec / dens if dens else np.nan})
sensitivity = pd.DataFrame(rows)

### 6.2. Results with τ = 0.5 (chosen values: k = 3, min. years = 2)

In [24]:
def grid(col, tau=0.5, fmt="{:.0f}"):
    t = sensitivity[sensitivity["tau"] == tau].pivot_table(index=["k", "min_years"], columns="period", values=col)
    return t.apply(lambda s: s.map(fmt.format))

grid("stable_ties")

period        P1   P2   P3   P4
k min_years                    
2 1           51  126  115   83
  2           15   70   65   51
  3            4   45   41   28
  4            3   26   21   21
3 1           74  194  172  148
  2           23  112   99  101
  3           10   74   66   60
  4            8   50   36   39
4 1          123  272  240  218
  2           52  155  151  145
  3           32  105  100   92
  4           25   74   53   63

In [25]:
grid("isolated")

period       P1  P2  P3  P4
k min_years                
2 1           4   1   0   2
  2          20   8   6   8
  3          29  16  15  15
  4          31  20  28  20
3 1           1   0   0   0
  2          14   6   4   2
  3          22  14  10   4
  4          25  15  20  10
4 1           1   0   0   0
  2           7   5   2   2
  3          12   9   6   4
  4          14  12  12   5

In [26]:
grid("reciprocity", fmt="{:.2f}")

period         P1    P2    P3    P4
k min_years                        
2 1          0.31  0.30  0.33  0.27
  2          0.40  0.26  0.28  0.24
  3          0.50  0.27  0.34  0.07
  4          0.67  0.08  0.29  0.10
3 1          0.30  0.32  0.30  0.27
  2          0.35  0.36  0.26  0.20
  3          0.20  0.35  0.24  0.07
  4          0.25  0.24  0.28  0.05
4 1          0.28  0.34  0.33  0.27
  2          0.23  0.37  0.29  0.19
  3          0.19  0.38  0.26  0.11
  4          0.08  0.32  0.26  0.10

In [27]:
grid("reciprocity_vs_random", fmt="{:.1f}")

period          P1    P2    P3   P4
k min_years                        
2 1            7.3   5.4   6.2  6.3
  2           31.7   8.3   9.2  9.1
  3          148.8  13.4  18.0  5.1
  4          264.4   6.7  29.4  9.0
3 1            4.8   3.7   3.8  3.6
  2           18.0   7.2   5.7  3.9
  3           23.8  10.7   7.9  2.2
  4           37.2  10.8  16.7  2.6
4 1            2.7   2.8   3.0  2.4
  2            5.3   5.4   4.2  2.6
  3            7.0   8.2   5.6  2.3
  4            3.8   9.9  10.8  3.0

### 6.3. Stable Ties with τ = 0.33

In [28]:
grid("stable_ties", tau=0.33)

period        P1   P2   P3   P4
k min_years                    
2 1           93  176  201  158
  2           42   99   98   93
  3           28   64   44   44
  4           23   34   21   21
3 1          151  276  296  255
  2           78  160  156  163
  3           59  103   72   80
  4           53   60   36   39
4 1          234  373  393  389
  2          137  221  231  255
  3          106  142  108  130
  4           95   84   53   63

### 6.4. What We Chose and Why

| Constant | Chosen | Tested | Why |
|---|---|---|---|
| **k** | **3** | 2, 4 | The 12, 10 and 8 points are the "headline" preferences of a vote. k = 2 keeps very few ties in P1 (15); k = 4 already includes mid-table points (7), which are closer to noise. |
| **τ** | **0.5** | 0.33 | "In at least half of the possible years" is a natural definition of a consistent preference. 0.33 is kept as a robustness check. |
| **min. years** | **2** | 1, 3, 4 | 1 lets single-vote artefacts in (P1: 74 ties instead of 23). 3 or 4 empty the short periods: with 7 finals in P3, requiring 4 years is almost the same as τ = 0.57, and P1 is left with 3–10 ties and up to 31 of 35 countries isolated. |

**What does not depend on these choices:** in all twelve combinations of k and min. years, reciprocity is well above the random expectation, and P4 always has the lowest reciprocity.

**What does:** the binary network of P1 is fragile (its reciprocity ranges from 0.08 to 0.67 across combinations), so binary results for P1 should be read with caution. The effect of k on the weighted networks (modularity, blocs, core–periphery) is tested in notebook 03.

## 7. P4 by Voting Channel: Jury and Televote

Since 2016 every country gives **two independent sets** of 12-10-8-…-1 points: one from a professional jury and one from the public (televote). Both refer to the same songs, year and participants, so any difference between the two networks can only come from **who votes**. This is the most direct test available of whether juries and the public vote differently (H2).

The networks are built with exactly the same function and settings as the main P4 network, using `jury_points` or `tele_points` instead of `total_points`. Within one channel a voter never gives the same points twice, so the top 3 has no ties. Notebook 04 analyses these networks.

### 7.1. Votes per Channel

In [29]:
cols_ch = ["year", "from_country_id", "to_country_id", "jury_points", "tele_points", "total_points"]
df_ch = pd.concat([df_spijk[(df_spijk["round"] == "final") & df_spijk["year"].between(2016, 2023)][cols_ch],
                   df_josago[(df_josago["round"] == "final") & (df_josago["year"] == 2024)][cols_ch]], ignore_index=True)
for c in ["from_country_id", "to_country_id"]:
    df_ch[c] = df_ch[c].astype(str).str.strip().str.upper()
df_ch = df_ch[~df_ch["from_country_id"].isin(["WLD", "ROW"]) & ~df_ch["to_country_id"].isin(["WLD", "ROW"])]
df_ch["from_iso3"] = df_ch["from_country_id"].map(ISO2_TO_ISO3)
df_ch["to_iso3"] = df_ch["to_country_id"].map(ISO2_TO_ISO3)
for c in ["jury_points", "tele_points", "total_points"]:
    df_ch[c] = pd.to_numeric(df_ch[c], errors="coerce").fillna(0)

assert df_ch[["from_iso3", "to_iso3"]].notna().all().all()
assert (df_ch["jury_points"] + df_ch["tele_points"] == df_ch["total_points"]).all()
print(f"{len(df_ch)} vote rows, finals: {sorted(df_ch['year'].unique())}")

8244 vote rows, finals: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


### 7.2. Build and Export the Channel Networks

In [30]:
p4_y0, p4_y1 = ERAS["P4"]
channel_results = {}
for channel, col in {"jury": "jury_points", "televote": "tele_points"}.items():
    adj_c, _, nodes_c, topk_years_c, co_years_c = compute_period_network(df_ch.assign(total_points=df_ch[col]), "P4", p4_y0, p4_y1, k=3)
    bin_c = ((adj_c >= 0.5) & (topk_years_c >= 2)).astype(int)
    bin_c = bin_c.where(~np.eye(len(bin_c), dtype=bool), 0)
    channel_results[channel] = adj_c
    adj_c.to_csv(os.path.join(DATA_CHANNELS, f"P4_{channel}_adj_topk3.csv"))
    bin_c.to_csv(os.path.join(DATA_CHANNELS, f"P4_{channel}_adj_binary_t05.csv"))
    co_years_c.to_csv(os.path.join(DATA_CHANNELS, f"P4_{channel}_co_years.csv"))
    print(f"{channel}: {len(nodes_c)} countries, {bin_c.values.sum()} stable ties")

jury: 45 countries, 71 stable ties
televote: 45 countries, 111 stable ties


Check: both channels have the same countries as the main P4 network.

In [31]:
p4_nodes = list(main_results["P4"][0].index)
print(all(list(channel_results[c].index) == p4_nodes for c in channel_results))

True


## 8. Country-Pair Attributes (CEPII)

To test whether geography and language explain the votes (H2), every pair of countries gets two binary attributes, stored as matrices with the same countries and order as the networks:

- **contig** = 1 if the two countries share a land border;
- **comlang_off** = 1 if they share an official language.

CEPII GeoDist predates some of the countries in the contest, so a few entries are completed by hand inside the function below:

- Romania's legacy code `ROM` is renamed `ROU`;
- **Monaco** (missing from CEPII): border with France, French as official language;
- **Serbia**, **Serbia and Montenegro** and **Montenegro** (missing from CEPII): borders and Serbo-Croatian language ties with their neighbours.

In [32]:
def build_cepii_matrices(all_nodes):
    """
    Extract contig and comlang_off from CEPII dist_cepii.dta aligned to all_nodes index.
    Standardizes CEPII codes:
    ROM -> ROU
    YUG -> YUG (retained for P1 historical Yugoslavia)
    SCG -> Serbia and Montenegro (P2) shares contiguity/language with YUG/SRB
    SRB / MNE -> Serbia and Montenegro post-dissolution
    MCO -> Monaco borders France (FRA), official French
    """
    cepii_path = os.path.join(DATA_RAW, "dist_cepii.dta")
    df_cepii, _ = pyreadstat.read_dta(cepii_path)

    # Standardize CEPII codes
    df_cepii["iso_o"] = df_cepii["iso_o"].replace({"ROM": "ROU"})
    df_cepii["iso_d"] = df_cepii["iso_d"].replace({"ROM": "ROU"})

    contig_map = {}
    comlang_map = {}

    for _, row in df_cepii.iterrows():
        o = row["iso_o"]
        d = row["iso_d"]
        c = int(row["contig"])
        l = int(row["comlang_off"])
        contig_map[(o, d)] = c
        comlang_map[(o, d)] = l

    # Specific historical / microstate completions based on official geography and CEPII GeoDist definitions
    # Monaco (MCO): borders FRA (contig=1), official language French (comlang_off=1 with FRA, BEL, CHE, LUX)
    for target in all_nodes:
        contig_map[("MCO", target)] = 1 if target == "FRA" else contig_map.get(("MCO", target), 0)
        contig_map[(target, "MCO")] = 1 if target == "FRA" else contig_map.get((target, "MCO"), 0)
        
        mco_lang = 1 if target in ["FRA", "BEL", "CHE", "LUX"] else 0
        comlang_map[("MCO", target)] = mco_lang
        comlang_map[(target, "MCO")] = mco_lang

    # Serbia (SRB) & Serbia and Montenegro (SCG):
    # Borders: BIH, HRV, HUN, ROU, BGR, MKD, ALB, MNE
    # Language: Serbo-Croatian with BIH, HRV, MNE (comlang_off=1)
    srb_borders = ["BIH", "HRV", "HUN", "ROU", "BGR", "MKD", "ALB", "MNE"]
    srb_lang = ["BIH", "HRV", "MNE", "SCG", "YUG", "SRB"]
    for s_code in ["SRB", "SCG"]:
        for target in all_nodes:
            if target != s_code:
                if target in srb_borders:
                    contig_map[(s_code, target)] = 1
                    contig_map[(target, s_code)] = 1
                if target in srb_lang:
                    comlang_map[(s_code, target)] = 1
                    comlang_map[(target, s_code)] = 1

    # Montenegro (MNE):
    # Borders: BIH, SRB, SCG, ALB, HRV
    # Language: comlang_off=1 with SRB, SCG, BIH, HRV, YUG
    mne_borders = ["BIH", "SRB", "SCG", "ALB", "HRV"]
    mne_lang = ["BIH", "SRB", "SCG", "HRV", "YUG", "MNE"]
    for target in all_nodes:
        if target != "MNE":
            if target in mne_borders:
                contig_map[("MNE", target)] = 1
                contig_map[(target, "MNE")] = 1
            if target in mne_lang:
                comlang_map[("MNE", target)] = 1
                comlang_map[(target, "MNE")] = 1

    # Build square DataFrames indexed by all_nodes
    mat_contig = pd.DataFrame(0, index=all_nodes, columns=all_nodes)
    mat_comlang = pd.DataFrame(0, index=all_nodes, columns=all_nodes)

    for u in all_nodes:
        for v in all_nodes:
            if u != v:
                mat_contig.loc[u, v] = contig_map.get((u, v), 0)
                mat_comlang.loc[u, v] = comlang_map.get((u, v), 0)

    return mat_contig, mat_comlang

### 8.1. Matrices for All Countries

In [33]:
mat_contig_all, mat_comlang_all = build_cepii_matrices(all_nodes)
mat_contig_all.to_csv(os.path.join(DATA_PROC, "cepii_contig_all.csv"))
mat_comlang_all.to_csv(os.path.join(DATA_PROC, "cepii_comlang_all.csv"))

### 8.2. Matrices Aligned with Each Period

In [34]:
for period, nodes in period_nodes.items():
    mat_contig_p = mat_contig_all.loc[nodes, nodes]
    mat_comlang_p = mat_comlang_all.loc[nodes, nodes]
    mat_contig_p.to_csv(os.path.join(DATA_PROC, f"{period}_cepii_contig.csv"))
    mat_comlang_p.to_csv(os.path.join(DATA_PROC, f"{period}_cepii_comlang.csv"))
    print(f"  Aligned CEPII matrices for {period} ({len(nodes)}x{len(nodes)})")

  Aligned CEPII matrices for P1 (35x35)
  Aligned CEPII matrices for P2 (48x48)
  Aligned CEPII matrices for P3 (47x47)
  Aligned CEPII matrices for P4 (45x45)


## 9. Dataset Summary Table

Descriptive statistics of the networks of each period, exported as a LaTeX table for the report (`paper/table_dataset_summary.tex`).

- **Finals**: number of Grand Finals in the period;
- **N**: countries that voted or competed;
- **Weighted edges / mean W**: pairs with weight > 0 and their average weight;
- **Bin t ≥ 0.5 / t ≥ 0.33**: stable ties in the main and robustness binary networks;
- **Contig % / ComLang %**: share of country pairs that share a border / an official language.

In [35]:
SUMMARY_ERAS = {
    "P1": {"span": "1975--1996", "desc": "Jury only", "y0": 1975, "y1": 1996},
    "P2": {"span": "1997--2008", "desc": "Televote rollout", "y0": 1997, "y1": 2008},
    "P3": {"span": "2009--2015", "desc": "50/50 Jury and Televote", "y0": 2009, "y1": 2015},
    "P4": {"span": "2016--2024", "desc": "Separate voting (combined)", "y0": 2016, "y1": 2024},
}

In [36]:
def count_editions(y0, y1):
    """Count number of contest finals in [y0, y1] from raw votes."""
    df_spijk = pd.read_csv(os.path.join(DATA_RAW, "votes_spijkervet.csv"))
    df_josago = pd.read_csv(os.path.join(DATA_RAW, "votes_josago.csv"))

    years_spijk = set(df_spijk[(df_spijk["year"] >= y0) & (df_spijk["year"] <= y1) & (df_spijk["round"] == "final")]["year"])
    years_josago = set(df_josago[(df_josago["year"] >= y0) & (df_josago["year"] <= y1) & (df_josago["round"] == "final")]["year"])
    return len(years_spijk | years_josago)

### 9.1. Compute the Statistics

In [37]:
rows = []

print(f"{'Period':<6} | {'Span':<11} | {'Finals':<6} | {'Nodes':<5} | {'Edges (k=3)':<11} | {'Mean W':<8} | {'Bin t>=0.5':<10} | {'Bin t>=0.33':<11} | {'Contig %':<8} | {'ComLang %':<9}")
print("-" * 105)

for period, meta in SUMMARY_ERAS.items():
    n_finals = count_editions(meta["y0"], meta["y1"])

    # 1. Weighted adj (top-k=3)
    adj_topk3 = pd.read_csv(os.path.join(DATA_PROC, f"{period}_adj_topk3.csv"), index_col=0)
    n_nodes = len(adj_topk3)

    # Non-zero directed edges and mean weight
    w_values = adj_topk3.values[~np.eye(n_nodes, dtype=bool)]
    nonzero_w = w_values[w_values > 0]
    n_nonzero_edges = len(nonzero_w)
    mean_weight = float(np.mean(nonzero_w)) if len(nonzero_w) > 0 else 0.0

    # 2. Binary t >= 0.5
    adj_bin_05 = pd.read_csv(os.path.join(DATA_PROC, f"{period}_adj_binary_t05.csv"), index_col=0)
    n_bin_05 = int(adj_bin_05.values.sum())

    # 3. Binary t >= 0.33 (robustness)
    adj_bin_033 = pd.read_csv(os.path.join(DATA_ROBUST, f"{period}_adj_binary_t033.csv"), index_col=0)
    n_bin_033 = int(adj_bin_033.values.sum())

    # 4. CEPII contig & comlang
    mat_contig = pd.read_csv(os.path.join(DATA_PROC, f"{period}_cepii_contig.csv"), index_col=0)
    mat_comlang = pd.read_csv(os.path.join(DATA_PROC, f"{period}_cepii_comlang.csv"), index_col=0)

    total_pairs = n_nodes * (n_nodes - 1)
    contig_pairs = int(mat_contig.values[~np.eye(n_nodes, dtype=bool)].sum())
    comlang_pairs = int(mat_comlang.values[~np.eye(n_nodes, dtype=bool)].sum())

    pct_contig = (contig_pairs / total_pairs * 100) if total_pairs > 0 else 0.0
    pct_comlang = (comlang_pairs / total_pairs * 100) if total_pairs > 0 else 0.0

    rows.append({
        "Period": period,
        "Description": meta["desc"],
        "Span": meta["span"],
        "Finals": n_finals,
        "Nodes": n_nodes,
        "WeightedEdges": n_nonzero_edges,
        "MeanWeight": mean_weight,
        "BinT05": n_bin_05,
        "BinT033": n_bin_033,
        "ContigPct": pct_contig,
        "ComLangPct": pct_comlang,
    })

    print(f"{period:<6} | {meta['span']:<11} | {n_finals:<6} | {n_nodes:<5} | {n_nonzero_edges:<11} | {mean_weight:<8.3f} | {n_bin_05:<10} | {n_bin_033:<11} | {pct_contig:<7.2f}% | {pct_comlang:<8.2f}%")

Period | Span        | Finals | Nodes | Edges (k=3) | Mean W   | Bin t>=0.5 | Bin t>=0.33 | Contig % | ComLang %
---------------------------------------------------------------------------------------------------------
P1     | 1975--1996  | 22     | 35    | 518         | 0.261    | 23         | 78          | 7.73   % | 5.88    %
P2     | 1997--2008  | 12     | 48    | 615         | 0.387    | 112        | 160         | 8.33   % | 3.19    %
P3     | 2009--2015  | 7      | 47    | 536         | 0.410    | 99         | 156         | 7.86   % | 3.05    %
P4     | 2016--2024  | 8      | 45    | 686         | 0.321    | 101        | 163         | 7.78   % | 3.64    %


In [38]:
df_summary = pd.DataFrame(rows)
df_summary

,Period,Description,Span,Finals,Nodes,WeightedEdges,MeanWeight,BinT05,BinT033,ContigPct,ComLangPct
0,P1,Jury only,1975--1996,22,35,518,0.261338,23,78,7.731092,5.882353
1,P2,Televote rollout,1997--2008,12,48,615,0.387271,112,160,8.333333,3.191489
2,P3,50/50 Jury and Televote,2009--2015,7,47,536,0.410488,99,156,7.863090,3.052729
3,P4,Separate voting (combined),2016--2024,8,45,686,0.321196,101,163,7.777778,3.636364


### 9.2. Export the LaTeX Table

In [39]:
# Generate LaTeX table
latex_lines = [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{Eurovision Voting Networks Dataset Summary across Eras (1975--2024)}",
    r"\label{tab:dataset_summary}",
    r"\small",
    r"\begin{tabular}{l l c c c c c c c c}",
    r"\hline\hline",
    r"Period & Voting System & Years & Finals & $N$ & $E_{\text{weighted}}$ & Mean $W$ & $E_{t \ge 0.5}$ & $E_{t \ge 0.33}$ & Contig. (\%) & ComLang (\%) \\",
    r"\hline",
]

for r in rows:
    line = (
        f"{r['Period']} & {r['Description']} & {r['Span']} & {r['Finals']} & {r['Nodes']} & "
        f"{r['WeightedEdges']} & {r['MeanWeight']:.3f} & {r['BinT05']} & {r['BinT033']} & "
        f"{r['ContigPct']:.1f}\\% & {r['ComLangPct']:.1f}\\% \\\\"
    )
    latex_lines.append(line)

latex_lines.extend([
    r"\hline\hline",
    r"\end{tabular}",
    r"\begin{tablenotes}",
    r"\footnotesize",
    r"\item \textit{Note:} $N$ represents active Grand Final participants/voters in the era; $E_{\text{weighted}}$ denotes directed ties where $W(A \to B) > 0$ for top-$k=3$; $E_{t \ge 0.5}$ and $E_{t \ge 0.33}$ are binarized edges at the main ($0.50$) and robustness ($0.33$) persistence thresholds requiring at least two top-$k$ voting occurrences ($\text{top-}k\text{ years} \ge 2$) to eliminate single-year artifact ties from late joiners; Contig. and ComLang show the percentage of directed dyads sharing a land border or official language from CEPII GeoDist.",
    r"\end{tablenotes}",
    r"\end{table}",
])

latex_content = "\n".join(latex_lines) + "\n"
latex_path = os.path.join(PAPER_DIR, "table_dataset_summary.tex")
with open(latex_path, "w", encoding="utf-8") as f:
    f.write(latex_content)

print(f"\nLaTeX table successfully exported to: {latex_path}")


LaTeX table successfully exported to: c:\Users\dudab\Projects\eurovision-sna\paper\table_dataset_summary.tex


## 10. Outputs

| Folder | Files | Used in |
|---|---|---|
| `data/processed/` | `{period}_adj_topk3.csv`, `{period}_edgelist_topk3.csv`, `{period}_adj_binary_t05.csv`, `{period}_cepii_*.csv`, `cepii_*_all.csv` | notebooks 03, 04 |
| `data/processed/robustness/` | k = 2 and k = 4, rank weights, τ = 0.33, unconstrained binary, P1a / P1b | notebook 03 |
| `data/processed/p4_channels/` | jury and televote networks of P4 | notebook 04 |
| `paper/` | `table_dataset_summary.tex` | report |